# DEL PARQUET ANTERIOR AGREGAR ONT_MODEL Y SOFTWARE VERSION

In [1]:
import pandas as pd #Importar librería pandas
import os #Importar módulo sistema
import glob #Importar módulo búsqueda archivos
import numpy as np #Importar librería numérica
import json #Importar módulo json
from datetime import datetime #Importar formateo de fechas

# Cargar configuración desde JSON
with open('config_etl_silver.json', 'r') as file: #Abrir archivo JSON
    config = json.load(file) #Cargar datos del JSON

extr_anio = config['etl_params']['anio'] #Definir año a procesar desde JSON
extr_mes = config['etl_params']['mes'] #Definir mes a procesar desde JSON

# Construir el formato AAMM (ej. 2026, mes 2 -> 2602)
str_yymm = datetime(extr_anio, extr_mes, 1).strftime('%y%m') #Formatear fecha

RUTA_SILVER = "datalake/silver/rx_avg" #Definir ruta silver
RUTA_SILVER_MODEL_SW = "datalake/silver/ont-model_sw-version" #Definir ruta ont-model_sw-version silver
RUTA_BRONZE_OLT = "datalake/bronze/huawei_inventory" #Definir ruta olt

if not os.path.exists(RUTA_SILVER_MODEL_SW): #Verificar existencia directorio
    os.makedirs(RUTA_SILVER_MODEL_SW) #Crear directorio si falta

# Archivo Base (Dinámico con AAMM)
FILE_INPUT = f"{RUTA_SILVER}/master_rx+services_silver_{str_yymm}.parquet" #Definir entrada dinámica

# Archivos Salida Integrado Final (Dinámicos con AAMM)
FILE_OUTPUT_PARQUET = f"{RUTA_SILVER_MODEL_SW}/master_model+sw_silver_{str_yymm}.parquet" #Definir salida parquet dinámica
FILE_OUTPUT_CSV = f"{RUTA_SILVER_MODEL_SW}/master_model+sw_silver_{str_yymm}.csv" #Definir salida csv dinámica

def integrar_olt_final(): #Definir función principal
    print(f"Iniciando Integración Final con OLT para el periodo {str_yymm}...") #Imprimir inicio

    try: #Iniciar bloque manejo de errores
        print("   Cargando Master Silver Enriched...") #Imprimir estado carga
        df_base = pd.read_parquet(FILE_INPUT) #Cargar parquet base
        
        #Normalizar llave Contrato
        df_base['CONTRATO'] = df_base['CONTRATO'].astype(str).str.strip() #Limpiar contrato base
        
        #Detectar columna de Modem correcta (Puede ser MODEMS, MODEM(S) o MODEM(S)_x)
        col_modem = next((c for c in ['MODEM(S)', 'MODEMS', 'MODEM(S)_x'] if c in df_base.columns), None) #Buscar columna modem
        if col_modem: #Validar existencia
            print(f"   Columna de Módem detectada: {col_modem}") #Imprimir columna
            df_base['key_serial_base'] = df_base[col_modem].astype(str).str.strip().str.upper().str[-8:] #Crear llave serial 8 digitos
        else:
            print("   [ALERTA] No se encontró columna de Módem. El cruce por serial fallará.") #Imprimir alerta
            df_base['key_serial_base'] = "INVALID" #Asignar valor dummy

        print("   Cargando Inventario OLT...") #Imprimir estado olt
        # Listar SOLO los archivos de OLT que correspondan al mes procesado
        archivos_olt = glob.glob(f"{RUTA_BRONZE_OLT}/inventory_master_*.parquet") #Listar archivos olt filtrados
        
        if not archivos_olt: #Validar existencia
            print(f"   [ERROR] No hay archivos de OLT para el mes {str_yymm}.") #Imprimir error
            return #Salir

        #Leer y concatenar inventario
        df_olt = pd.concat((pd.read_parquet(f) for f in archivos_olt), ignore_index=True) #Concatenar olt
        
        #Seleccionar columnas a integrar (SERIAL_NUMBER, DESCRIPTION, ONT_MODEL, SOFTWARE_VERSION)
        cols_olt = ['serial_number', 'description', 'ont_model', 'software_version'] #Definir columnas olt
        df_olt = df_olt[cols_olt].copy() #Filtrar columnas
        
        #Normalizar llaves de OLT
        df_olt['description'] = df_olt['description'].astype(str).str.strip() #Limpiar description (contrato)
        df_olt['key_serial_olt'] = df_olt['serial_number'].astype(str).str.strip().str.upper().str[-8:] #Crear llave serial 8 digitos

        #Eliminar duplicados en OLT para evitar multiplicar filas (priorizando si hay repetidos)
        df_olt = df_olt.drop_duplicates(subset=['description'], keep='last') #Deduplicar por contrato
        
        print("   Ejecutando Cruce (Prioridad: Contrato -> Serial)...") #Imprimir estado cruce
        
        #Renombrar columnas OLT para el merge
        df_match1 = pd.merge( #Ejecutar primer merge
            df_base, 
            df_olt.rename(columns={'description': 'CONTRATO_MATCH'}), #Renombrar para evitar choque
            left_on='CONTRATO', 
            right_on='CONTRATO_MATCH', 
            how='left'
        ) #Fin primer merge

        #Separar los que NO cruzaron (donde ont_model es nulo)
        df_match2 = pd.merge( #Ejecutar segundo merge
            df_base,
            df_olt,
            left_on='key_serial_base',
            right_on='key_serial_olt',
            how='left',
            suffixes=('', '_serial')
        ) #Fin segundo merge

        print("   Consolidando columnas...") #Imprimir estado consolidación
        
        #Inicializar columnas finales en el dataframe principal (df_match1)
        cols_target = ['serial_number', 'ont_model', 'software_version'] #Columnas objetivo
        
        for col in cols_target: #Iterar columnas
            
            #Asegurar que las columnas existan en match1 (pueden ser NaN)
            if col not in df_match1.columns: df_match1[col] = np.nan #Crear si falta
            
            #Tomar valores de match2
            vals_match2 = df_match2[col] if col in df_match2.columns else df_match2[col + '_serial'] #Obtener valor match2
            
            #Rellenar nulos
            df_match1[col] = df_match1[col].fillna(vals_match2) #Combinar resultados

        if 'description' not in df_match1.columns: #Verificar description
            vals_desc2 = df_match2['description'] #Obtener description match2
            df_match1['description'] = vals_desc2 #Asignar description match2
        else:
            df_match1['description'] = df_match1['description'].fillna(df_match2['description']) #Rellenar description

        #Eliminar columnas auxiliares de llaves
        cols_drop = ['key_serial_base', 'CONTRATO_MATCH', 'key_serial_olt'] #Columnas a borrar
        df_final = df_match1.drop(columns=[c for c in cols_drop if c in df_match1.columns]) #Borrar columnas
        
        #Estandarizar tipos
        for col in df_final.columns: #Iterar columnas
            if df_final[col].dtype == 'object': #Verificar tipo objeto
                df_final[col] = df_final[col].astype(str) #Convertir string

        print(f"   Guardando en {RUTA_SILVER_MODEL_SW}...") #Imprimir estado guardado
        df_final.to_parquet(FILE_OUTPUT_PARQUET, index=False) #Exportar parquet
        df_final.to_csv(FILE_OUTPUT_CSV, index=False, encoding='utf-8-sig') #Exportar csv

        print(f"[EXITO] Integración Final completada.") #Imprimir éxito
        print(f"Total Registros: {len(df_final)}") #Imprimir total
        print(f"Columnas Finales: {list(df_final.columns)}") #Imprimir columnas
        
        print("Muestra (Top 5):") #Imprimir cabecera muestra
        
        cols_show = ['CONTRATO', 'MODEMS' if 'MODEMS' in df_final.columns else col_modem, 'serial_number', 'ont_model', 'software_version'] #Seleccionar columnas
        cols_show = [c for c in cols_show if c in df_final.columns] #Validar existencia
        print(df_final[cols_show].head()) #Imprimir muestra filtrada para que sea más legible en consola
        
    except FileNotFoundError as e: #Capturar error archivo no encontrado
        print(f"[ERROR] Archivo base no encontrado: {e}") #Imprimir mensaje error
    except Exception as e: #Capturar otros errores
        print(f"[ERROR] Ha ocurrido un error inesperado: {e}") #Imprimir mensaje error

if __name__ == "__main__": #Verificar ejecución directa
    integrar_olt_final() #Ejecutar función principal

Iniciando Integración Final con OLT para el periodo 2602...
   Cargando Master Silver Enriched...
   Columna de Módem detectada: MODEM(S)_x
   Cargando Inventario OLT...
   Ejecutando Cruce (Prioridad: Contrato -> Serial)...
   Consolidando columnas...
   Guardando en datalake/silver/ont-model_sw-version...
[EXITO] Integración Final completada.
Total Registros: 424
Columnas Finales: ['CONTRATO', 'FECHA_PEDIDO', 'MOTIVO_PEDIDO', 'PAQUETE_x', 'DETALLE_PEDIDO1', 'DETALLE_PEDIDO2', 'FECHA_CUMPLIMIENTO', 'COD_TECNICO', 'ESTATUS_x', 'ZONA', 'MODEM(S)_x', 'ESTATUS_y', 'PAQUETE_y', 'FECHA_INSTALACION', 'MODEM(S)_y', 'DIAS_ANTIGUEDAD', 'DIAS_ATENCION', 'ont_id', 'rx_avg', 'serial_number', 'ont_model', 'software_version', 'description']
Muestra (Top 5):
  CONTRATO        MODEM(S)_x     serial_number     ont_model software_version
0    42295      XPONDD45BHEA  58504F4EDD45BAEA      MN-WIFI2           V1.7.1
1    45662      XPON1DE0044A  58504F4E1DE0044A       HG102AT           V1.7.1
2    38705  